In [66]:
#!/usr/bin/env python
# coding: utf-8


In [67]:
import pandas as pd

df = pd.read_parquet('/home/mschae/ETRI/ch2025_mACStatus.parquet')
print(len(df))
print((df['subject_id'] == 'id01').sum())



939896
94266


In [68]:
df_train = pd.read_csv('/home/mschae/ETRI/ch2025_metrics_train.csv')
print((df_train['subject_id'] == 'id01').sum())



41


In [69]:


import glob
import os
data_dir = r'/home/mschae/ETRI'
parquet_files = glob.glob(os.path.join(data_dir,'ch2025_*.parquet'))
lifelog_data = {}
for file_path in parquet_files:
    name = os.path.basename(file_path).replace('.parquet','').replace('ch2025_','')
    lifelog_data[name] = pd.read_parquet(file_path)
    print(f'Loaded: {name}, shape: {lifelog_data[name].shape}')
# 각 요소별로 데이터 나눈것 그러니까 parquet 읽어와서 dict으로 저장



Loaded: mWifi, shape: (76336, 3)
Loaded: wHr, shape: (382918, 3)
Loaded: mBle, shape: (21830, 3)
Loaded: mScreenStatus, shape: (939653, 3)
Loaded: mActivity, shape: (961062, 3)
Loaded: mUsageStats, shape: (45197, 3)
Loaded: wLight, shape: (633741, 3)
Loaded: mAmbience, shape: (476577, 3)
Loaded: mLight, shape: (96258, 3)
Loaded: wPedo, shape: (748100, 9)
Loaded: mGps, shape: (800611, 3)
Loaded: mACStatus, shape: (939896, 3)


In [70]:
metrics_train = pd.read_csv('/home/mschae/ETRI/ch2025_metrics_train.csv')
sample_submission = pd.read_csv('/home/mschae/ETRI/ch2025_submission_sample.csv')

# 날짜 처리
sample_submission['lifelog_date'] = pd.to_datetime(sample_submission['lifelog_date'])
test_keys = set(zip(sample_submission['subject_id'], sample_submission['lifelog_date'].dt.date))
# csv 파일 읽어와서 test_keys에 저장함 train용은 지금 따로 만들어야될듯



In [71]:


def split_test_train(df, subject_col = 'subject_id', timestamp_col = 'timestamp'):
    df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors='coerce')
    df = df.dropna(subset = [timestamp_col])
    df['date_only'] = df[timestamp_col].dt.date
    df['key'] = list(zip(df[subject_col], df['date_only']))

    test_df = df[df['key'].isin(test_keys)].drop(columns = ['date_only','key'])
    train_df = df[~df['key'].isin(test_keys)].drop(columns = ['date_only', 'key'])

    return test_df, train_df
# 일단 데이터를 나눈건데 test와 train으로 



In [72]:


for name, df in lifelog_data.items():
    print('분리중')
    test_df, train_df = split_test_train(df.copy())
    globals()[f'{name}_test'] = test_df
    globals()[f'{name}_train'] = train_df
    print(f'{name}_test -> {test_df.shape}, {name}_train -> {train_df.shape}')



분리중
mWifi_test -> (27467, 3), mWifi_train -> (48869, 3)
분리중
wHr_test -> (143311, 3), wHr_train -> (239607, 3)
분리중
mBle_test -> (8140, 3), mBle_train -> (13690, 3)
분리중
mScreenStatus_test -> (336160, 3), mScreenStatus_train -> (603493, 3)
분리중
mActivity_test -> (343579, 3), mActivity_train -> (617483, 3)
분리중
mUsageStats_test -> (16499, 3), mUsageStats_train -> (28698, 3)
분리중
wLight_test -> (233809, 3), wLight_train -> (399932, 3)
분리중
mAmbience_test -> (170453, 3), mAmbience_train -> (306124, 3)
분리중
mLight_test -> (34439, 3), mLight_train -> (61819, 3)
분리중
wPedo_test -> (288832, 9), wPedo_train -> (459268, 9)
분리중
mGps_test -> (287386, 3), mGps_train -> (513225, 3)
분리중
mACStatus_test -> (335849, 3), mACStatus_train -> (604047, 3)


In [73]:


import torch.nn as nn



In [74]:


class dataTransformer(nn.Module):
    def __init__(self, input_dim, d_model, nhead, num_layers):
        super().__init__()
        self.input_layer = nn.Linear(input_dim, d_model)
        self.input = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead)
        self.encoder = nn.TransformerEncoder(self.input, num_layers=num_layers)

    def forward(self, x):
        x = self.input_layer(x)               # (B, T, D)
        x = x.permute(1, 0, 2).contiguous()   # (T, B, D), contiguous for GPU
        encoded = self.encoder(x)
        return encoded


In [75]:


import torch
from collections import defaultdict



In [76]:
def process_mACStatus(df):
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['date'] = df['timestamp'].dt.date
    df = df.sort_values(['subject_id', 'timestamp'])
    results = {}
    output = defaultdict(list)
    subj_represent = {}
    for (subj, date), group in df.groupby(['subject_id', 'date']):
        charging = group['m_charging'].values
        month = group['timestamp'].dt.month
        day = group['timestamp'].dt.day
        hour = group['timestamp'].dt.hour
        minute = group['timestamp'].dt.minute
        second = group['timestamp'].dt.second
        feature = torch.stack([
            torch.tensor(charging, dtype=torch.float32),
            torch.tensor(month.values, dtype = torch.float32),
            torch.tensor(day.values, dtype = torch.float32),
            torch.tensor(hour.values, dtype = torch.float32),
            torch.tensor(minute.values, dtype=torch.float32),
            torch.tensor(second.values, dtype=torch.float32)
        ], dim=1).to('cuda' if torch.cuda.is_available() else 'cpu')
        results[(subj, date)] = feature.to('cuda' if torch.cuda.is_available() else 'cpu')
    model = dataTransformer(6,512,8,1).to('cuda' if torch.cuda.is_available() else 'cpu')
    for (subj, date), data in results.items():
        out = model(data.unsqueeze(0)) # batch 때문에 차원 늘리기
        pooled = out.mean(dim = 1).squeeze(0) #
        output[subj].append(pooled)
    for subj, vectors in output.items():
        subj_represent[subj] = torch.stack(vectors).mean(dim=0)
    return subj_represent



In [77]:
mACStatus_train_processing = process_mACStatus(mACStatus_train)



RuntimeError: CUDA error: CUBLAS_STATUS_EXECUTION_FAILED when calling `cublasSgemm( handle, opa, opb, m, n, k, &alpha, a, lda, b, ldb, &beta, c, ldc)`

In [ ]:


def process_mActivity(df):
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['date'] = df['timestamp'].dt.date
    df = df.sort_values(['subject_id', 'timestamp'])
    results = {}
    output = defaultdict(list)
    subj_represent = {}
    for (subj, date), group in df.groupby(['subject_id', 'date']):
        activity = group['m_activity'].values
        month = group['timestamp'].dt.month
        day = group['timestamp'].dt.day
        hour = group['timestamp'].dt.hour
        minute = group['timestamp'].dt.minute
        second = group['timestamp'].dt.second
        feature = torch.stack([
            torch.tensor(activity, dtype=torch.float32),
            torch.tensor(month.values, dtype = torch.float32),
            torch.tensor(day.values, dtype = torch.float32),
            torch.tensor(hour.values, dtype = torch.float32),
            torch.tensor(minute.values, dtype=torch.float32),
            torch.tensor(second.values, dtype=torch.float32)
        ], dim=1).to('cuda' if torch.cuda.is_available() else 'cpu')
        results[(subj, date)] = feature.to('cuda' if torch.cuda.is_available() else 'cpu')
    model = dataTransformer(6,512,8,1).to('cuda' if torch.cuda.is_available() else 'cpu')
    for (subj, date), data in results.items():
        out = model(data.unsqueeze(1)) # batch 때문에 차원 늘리기
        pooled = out.mean(dim = 1).squeeze(0) #
        output[subj].append(pooled)
    for subj, vectors in output.items():
        subj_represent[subj] = torch.stack(vectors).mean(dim=0)
    return subj_represent



In [ ]:


mActivity_train_processing = process_mActivity(mActivity_train)



In [ ]:


def process_mLight(df):
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['date'] = df['timestamp'].dt.date
    df = df.sort_values(['subject_id', 'timestamp'])
    results = {}
    output = defaultdict(list)
    subj_represent = {}
    for (subj, date), group in df.groupby(['subject_id', 'date']):
        light = group['m_light'].values
        month = group['timestamp'].dt.month
        day = group['timestamp'].dt.day
        hour = group['timestamp'].dt.hour
        minute = group['timestamp'].dt.minute
        second = group['timestamp'].dt.second
        feature = torch.stack([
            torch.tensor(light, dtype=torch.float32),
            torch.tensor(month.values, dtype = torch.float32),
            torch.tensor(day.values, dtype = torch.float32),
            torch.tensor(hour.values, dtype = torch.float32),
            torch.tensor(minute.values, dtype=torch.float32),
            torch.tensor(second.values, dtype=torch.float32)
        ], dim=1).to('cuda' if torch.cuda.is_available() else 'cpu')
        results[(subj, date)] = feature.to('cuda' if torch.cuda.is_available() else 'cpu')
    model = dataTransformer(6,512,8,1).to('cuda' if torch.cuda.is_available() else 'cpu')
    for (subj, date), data in results.items():
        out = model(data.unsqueeze(1)) # batch 때문에 차원 늘리기
        pooled = out.mean(dim = 1).squeeze(0) #
        output[subj].append(pooled)
    for subj, vectors in output.items():
        subj_represent[subj] = torch.stack(vectors).mean(dim=0)
    return subj_represent



In [ ]:


mLight_train_processing = process_mLight(mLight_train)



NameError: name 'process_mLight' is not defined

In [ ]:


class DeepLifelogModel(nn.Module):
    def __init__(self, in_dim, out_dim, hidden_dim):
        super().__init__()
        self.inputlayer1 = nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.ReLU())
        self.hiddenlayer = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU())
        self.outputlayer = nn.Sequential(nn.Linear(hidden_dim, out_dim), nn.ReLU())

    def forward(self, x):
        x = self.inputlayer1(x)
        x = self.hiddenlayer(x)
        out = self.outputlayer(x)
        return out



In [ ]:


model = DeepLifelogModel(3,6,256)



In [ ]:


final_df = {}

common_subjects = set(mACStatus_train_processing) & \
                  set(mActivity_train_processing) & \
                  set(mLight_train_processing)

for subj in sorted(common_subjects):
    vec1 = mACStatus_train_processing[subj]   
    vec2 = mActivity_train_processing[subj]   
    vec3 = mLight_train_processing[subj]     
    
    combined = torch.stack([vec1, vec2, vec3], dim=0) 
    final_df[subj] = combined.float()



In [ ]:


final_df['id01']

